In [ ]:
import os, re, json, time, random, unicodedata
import numpy as np
import pandas as pd
import torch
import fasttext
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# =========================================================== 1. Config
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_PATH = "/kaggle/input/datasets/peacepath/merged-intent-dataset/merged_intent_dataset.csv"
NEW_DATA_FILE = "/kaggle/input/datasets/peacepath/tabinda-dataset/updated_tabinda_fasttext_master_train_data.txt"
WORK_DIR = "./multilingual_intent"
TRANS_DIR = f"{WORK_DIR}/translations"
os.makedirs(TRANS_DIR, exist_ok=True)

TRANSLATION_MODEL = "facebook/nllb-200-distilled-600M"
BATCH_SIZE = 64
NUM_BEAMS = 1
MAX_NEW_TOKENS = 96
CHECKPOINT_EVERY = 20

LANGS = {"en": "eng_Latn", "ar": "arb_Arab", "fr": "fra_Latn", "ur": "urd_Arab"}
TARGET_LANGS = ["ar", "fr", "ur"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# your tuned hyperparameters — no autotune needed anymore
BEST_PARAMS = dict(
    lr=0.5981511727499647, dim=160, epoch=100, wordNgrams=2,
    minCount=1, minn=2, maxn=5, bucket=1173215, loss="softmax",
)
CONFIDENCE_THRESHOLD = 0.5

# =========================================================== 2. Load & clean English source
df_en = pd.read_csv(DATA_PATH)
df_en["query"] = df_en["query"].astype(str).str.strip()
df_en["intent"] = df_en["intent"].astype(str).str.strip()
df_en = df_en[(df_en["query"] != "") & (df_en["intent"] != "") & (df_en["intent"].str.lower() != "nan")]
df_en = df_en.drop_duplicates(subset=["query", "intent"])
conflicts = df_en.groupby(df_en["query"].str.lower())["intent"].nunique()
df_en = df_en[~df_en["query"].str.lower().isin(set(conflicts[conflicts > 1].index))]
df_en = df_en.reset_index(drop=True)
df_en["source_id"] = np.arange(len(df_en))
df_en["language"] = "en"
df_en = df_en[["source_id", "query", "intent", "language"]]

LABELS = sorted(df_en["intent"].unique())
print("clean English rows:", len(df_en), "| intents:", len(LABELS))

# =========================================================== 3. Translate (with resume)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tok = AutoTokenizer.from_pretrained(TRANSLATION_MODEL)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATION_MODEL, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE).eval()

def target_bos_id(tokenizer, lang_code):
    if hasattr(tokenizer, "lang_code_to_id"):
        return tokenizer.lang_code_to_id[lang_code]
    return tokenizer.convert_tokens_to_ids(lang_code)

@torch.no_grad()
def translate_batch(texts, src_lang, tgt_lang):
    tok.src_lang = src_lang
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
    out = mt_model.generate(**enc, forced_bos_token_id=target_bos_id(tok, tgt_lang),
                             max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
    return tok.batch_decode(out, skip_special_tokens=True)

def translate_column(texts, tgt_code, cache_path):
    cache = {}
    if os.path.exists(cache_path):
        cached = pd.read_csv(cache_path)
        cache = dict(zip(cached["source"].astype(str), cached["translation"].astype(str)))
    todo = [t for t in texts if t not in cache]
    for bi, i in enumerate(range(0, len(todo), BATCH_SIZE)):
        batch = todo[i:i + BATCH_SIZE]
        try:
            translations = translate_batch(batch, LANGS["en"], tgt_code)
        except RuntimeError:
            translations = [translate_batch([b], LANGS["en"], tgt_code)[0] for b in batch]
        cache.update(dict(zip(batch, translations)))
        if bi % CHECKPOINT_EVERY == 0 or i + BATCH_SIZE >= len(todo):
            pd.DataFrame({"source": list(cache.keys()), "translation": list(cache.values())}).to_csv(cache_path, index=False)
    return cache

unique_queries = df_en["query"].unique().tolist()
translated_frames = {"en": df_en.copy()}
for lang in TARGET_LANGS:
    mapping = translate_column(unique_queries, LANGS[lang], f"{TRANS_DIR}/{lang}.csv")
    frame = df_en.copy()
    frame["query"] = frame["query"].map(mapping)
    frame["language"] = lang
    translated_frames[lang] = frame

# =========================================================== 4. Merge + quality filter
AR_SCRIPT = re.compile(r"[\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]")
LATIN = re.compile(r"[A-Za-z]")

def script_ratio(text, pattern):
    letters = [c for c in str(text) if c.isalpha()]
    return sum(bool(pattern.match(c)) for c in letters) / len(letters) if letters else 0.0

frames = []
for lang, frame in translated_frames.items():
    f = frame.copy()
    f["query"] = f["query"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    f = f[(f["query"] != "") & (f["query"].str.lower() != "nan")]
    if lang != "en":
        expected = AR_SCRIPT if lang in ("ar", "ur") else LATIN
        keep = f["query"].apply(lambda t: script_ratio(t, expected) > 0.5)
        src_map = dict(zip(df_en["source_id"], df_en["query"].str.lower()))
        not_copy = f.apply(lambda r: r["query"].lower() != src_map.get(r["source_id"], ""), axis=1)
        f = f[keep & not_copy]
    frames.append(f[["source_id", "query", "intent", "language"]])

df_all = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["query", "intent"]).reset_index(drop=True)
print("merged rows:", len(df_all))

# =========================================================== 5. Normalize
AR_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
ZERO_WIDTH = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2066-\u2069\uFEFF]")
TATWEEL = "\u0640"
DIGIT_MAP = {}
for i in range(10):
    DIGIT_MAP[ord("\u0660") + i] = str(i)
    DIGIT_MAP[ord("\u06F0") + i] = str(i)
VARIANT_MAP = str.maketrans({
    "\u0623": "\u0627", "\u0625": "\u0627", "\u0622": "\u0627",
    "\u064A": "\u06CC", "\u06D2": "\u06CC", "\u0643": "\u06A9",
    "\u0647": "\u06C1", "\u06C3": "\u06C1", "\u0629": "\u06C1",
})
KEEP_CHARS = set("'&")

def normalize(text: str) -> str:
    t = unicodedata.normalize("NFKC", str(text)).strip().lower()
    t = ZERO_WIDTH.sub("", t)
    t = AR_DIACRITICS.sub("", t)
    t = t.replace(TATWEEL, "")
    t = t.translate(DIGIT_MAP)
    t = t.translate(VARIANT_MAP)
    t = "".join(c if (unicodedata.category(c)[0] in "LMN" or c in KEEP_CHARS) else " " for c in t)
    return re.sub(r"\s+", " ", t).strip()

df_all["text"] = df_all["query"].apply(normalize)
df_all = df_all[df_all["text"] != ""]
df_all = df_all.drop_duplicates(subset=["text", "intent"]).reset_index(drop=True)

# =========================================================== 6. Split by source query (no leakage)
source_meta = df_all.groupby("source_id")["intent"].first().reset_index()
train_ids, temp_ids = train_test_split(source_meta, test_size=0.20, random_state=SEED, stratify=source_meta["intent"])
valid_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=SEED, stratify=temp_ids["intent"])

train_df = df_all[df_all["source_id"].isin(set(train_ids["source_id"]))].reset_index(drop=True)
valid_df = df_all[df_all["source_id"].isin(set(valid_ids["source_id"]))].reset_index(drop=True)
test_df  = df_all[df_all["source_id"].isin(set(test_ids["source_id"]))].reset_index(drop=True)
print(f"rows -> train {len(train_df)}  valid {len(valid_df)}  test {len(test_df)}")

# =========================================================== 7. Fold in the extra hand-written dataset (train only)
def parse_fasttext_file(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            m = re.match(r"__label__(\S+)\s+(.*)", line)
            if m:
                rows.append({"intent": m.group(1), "query": m.group(2)})
    return pd.DataFrame(rows)

new_df = parse_fasttext_file(NEW_DATA_FILE)
if set(new_df["intent"]) - set(LABELS):
    raise ValueError(f"Unknown labels in new data: {set(new_df['intent']) - set(LABELS)}")
new_df["text"] = new_df["query"].apply(normalize)
new_df = new_df[new_df["text"] != ""].drop_duplicates(subset=["text", "intent"])

existing_valid_test = set(valid_df["text"]) | set(test_df["text"])
new_df = new_df[~new_df["text"].isin(existing_valid_test)].reset_index(drop=True)
print("extra training rows added:", len(new_df))

# =========================================================== 8. Write fastText files
TRAIN_FILE, VALID_FILE, TEST_FILE = f"{WORK_DIR}/train.txt", f"{WORK_DIR}/valid.txt", f"{WORK_DIR}/test.txt"

combined_train = pd.concat([train_df[["text", "intent"]], new_df[["text", "intent"]]], ignore_index=True)
combined_train = combined_train.sample(frac=1.0, random_state=SEED)

def write_ft(frame, path, cols=("text", "intent")):
    with open(path, "w", encoding="utf-8") as f:
        for text, intent in zip(frame[cols[0]], frame[cols[1]]):
            f.write(f"__label__{intent} {text}\n")

write_ft(combined_train, TRAIN_FILE)
write_ft(valid_df.sample(frac=1.0, random_state=SEED), VALID_FILE)
write_ft(test_df.sample(frac=1.0, random_state=SEED), TEST_FILE)
print(f"train.txt: {len(combined_train)} lines")

# =========================================================== 9. Train final model (fixed hyperparameters)
t0 = time.time()
model = fasttext.train_supervised(input=TRAIN_FILE, thread=os.cpu_count(), seed=SEED, verbose=1, **BEST_PARAMS)
print(f"trained in {time.time() - t0:.1f}s")

# =========================================================== 10. Evaluate
def predict_frame(ft_model, frame):
    labels, probs = ft_model.predict(frame["text"].tolist(), k=1)
    return [l[0].replace("__label__", "") for l in labels], [p[0] for p in probs]

y_true = test_df["intent"].tolist()
y_pred, y_conf = predict_frame(model, test_df)

print("test accuracy   :", round(accuracy_score(y_true, y_pred), 4))
print("test macro F1   :", round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4))

results_df = pd.DataFrame({"language": test_df["language"], "actual": y_true, "predicted": y_pred})
results_df["correct"] = results_df["actual"] == results_df["predicted"]
print(results_df.groupby("language")["correct"].mean().round(4))
print()
print(classification_report(y_true, y_pred, labels=LABELS, digits=4, zero_division=0))

# =========================================================== 11. Quantize for deployment
model.save_model(f"{WORK_DIR}/intent_model.bin")
model.quantize(input=TRAIN_FILE, qnorm=True, retrain=True, cutoff=200000)
model.save_model(f"{WORK_DIR}/intent_model.ftz")

# =========================================================== 12. Save config
config = {
    "labels": LABELS,
    "best_params": {k: str(v) for k, v in BEST_PARAMS.items()},
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "test_accuracy": float(accuracy_score(y_true, y_pred)),
    "test_macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    "rows": {"train": len(combined_train), "valid": len(valid_df), "test": len(test_df)},
}
with open(f"{WORK_DIR}/model_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

# =========================================================== 13. Inference
def predict(queries, k=1, threshold=CONFIDENCE_THRESHOLD, ft_model=None):
    ft_model = ft_model or model
    single = isinstance(queries, str)
    raw = [queries] if single else list(queries)
    batch = [normalize(q) for q in raw]
    labels, probs = ft_model.predict(batch, k=k)
    out = []
    for lab, pr in zip(labels, probs):
        preds = [(l.replace("__label__", ""), round(float(p), 4)) for l, p in zip(lab, pr)]
        if k == 1:
            intent, conf = preds[0]
            out.append((intent if conf >= threshold else None, conf))
        else:
            out.append(preds)
    return out[0] if single else out

for q in ["block my card please", "je veux annuler ma carte",
          "کارڈ بلاک کرنا ہے", "أريد تجميد بطاقتي"]:
    intent, conf = predict(q)
    print(f"{conf:.3f}  {intent or 'UNCERTAIN':<20}  {q}")